# 00 — Baseline: Auditoría, Embeddings BETO-STS, Relabeling k-NN y MLP

Notebook maestro para:
1. Verificar integridad de `particiones/` (`audit_splits.py`).
2. Generar (o reutilizar desde Google Drive) los embeddings congelados **BETO-STS** (`hiiamsid/sentence_similarity_spanish_es`, 768 dims).
3. Ejecutar el relabeling conservador `suffer -> control` (`J=10, K=5`, `exclude_same_subject=False`) por trastorno (`Ansiedad`, `Depresion`, `TCA`).
4. Entrenar el MLP baseline con 5 semillas (`11, 23, 37, 51, 79`) comparando `initial_labels` vs `knn_include_same_subject`.
5. Guardar checkpoints en Drive (`STORAGE_ROOT`) y exportar métricas, resúmenes y **logits/probabilidades por mensaje en orden cronológico** (`dev_message_predictions.csv`) para RQ1 y RQ2.

In [3]:
# 1) Montaje de Google Drive y detección automática de rutas
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Opcional en Colab si el repo no está en Drive: clona desde GitHub en /content/dm_project
REPO_URL = 'https://github.com/Urkibiris05/dm_project.git'
GIT_BRANCH = 'rq_larra'
GITHUB_TOKEN = ''  # Rellenar solo si el repo de GitHub es privado

def _is_repo_root(p: Path) -> bool:
    return (p / 'particiones').is_dir() and (p / 'rq_larra' / 'rq_utils.py').is_file()

def _find_repo() -> Path | None:
    candidates = [Path('/content/dm_project'), Path.cwd().resolve(), *Path.cwd().resolve().parents]
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.is_dir():
        candidates.append(drive_root / 'dm_project')
        for pattern in ('*', '*/*', '*/*/*'):
            for p in drive_root.glob(pattern):
                if p.is_dir() and p.name == 'dm_project':
                    candidates.append(p)
    return next((p for p in candidates if _is_repo_root(p)), None)

PROJECT_ROOT = _find_repo()

if PROJECT_ROOT is None and IN_COLAB:
    clone_dir = Path('/content/dm_project')
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@') if GITHUB_TOKEN else REPO_URL
    if not clone_dir.exists():
        print(f'Repo no encontrado en Drive; intentando clonar rama {GIT_BRANCH} en {clone_dir}...')
        res = subprocess.run(['git', 'clone', '-b', GIT_BRANCH, clone_url, str(clone_dir)], text=True, capture_output=True)
        if res.returncode != 0:
            print('Aviso git clone:', res.stderr.strip())
    PROJECT_ROOT = _find_repo()

if PROJECT_ROOT is None:
    drive_items = sorted(p.name for p in Path('/content/drive/MyDrive').iterdir())[:25] if Path('/content/drive/MyDrive').is_dir() else []
    raise FileNotFoundError(
        'No se encontró dm_project con particiones/ y rq_larra/rq_utils.py.\n'
        f'Carpetas vistas en MyDrive: {drive_items}\n'
        'Soluciones: (1) Sube la carpeta dm_project a Google Drive (p. ej. MyDrive/dm_project), o '
        '(2) haz git push origin rq_larra desde tu PC (y rellena GITHUB_TOKEN si el repo es privado).'
    )

if str(PROJECT_ROOT / 'rq_larra') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'rq_larra'))

print('PROJECT_ROOT:', PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo no encontrado en Drive; intentando clonar rama rq_larra en /content/dm_project...
Aviso git clone: Cloning into '/content/dm_project'...
fatal: Remote branch rq_larra not found in upstream origin


FileNotFoundError: No se encontró dm_project con particiones/ y rq_larra/rq_utils.py.
Carpetas vistas en MyDrive: ['BD provisional.drawio', 'Horas practicas.gsheet', 'Login.drawio', 'Math', 'Programming', 'Propuestas Baseline:.gdoc', 'Untitled document.gdoc', 'bash']
Soluciones: (1) Sube la carpeta dm_project a Google Drive (p. ej. MyDrive/dm_project), o (2) haz git push origin rq_larra desde tu PC (y rellena GITHUB_TOKEN si el repo es privado).

In [ ]:
# 2) Dependencias (solo instala si faltan en el entorno)
if IN_COLAB:
    import importlib.util
    import subprocess
    if importlib.util.find_spec('sentence_transformers') is None:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            'sentence-transformers', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'scipy'
        ])

In [ ]:
import shutil
import torch
import matplotlib.pyplot as plt
import pandas as pd
import rq_utils

# --- Configuración reproducible del experimento ---
DISORDERS = ('Ansiedad', 'Depresion', 'TCA')
SEEDS = (11, 23, 37, 51, 79)
NEIGHBORS_J = 10
MIN_CONTROL_K = 5
RELABEL_MODE = 'single_pass'   # 'single_pass' o 'iterative'
MAX_ROUNDS = None
BATCH_SIZE_EMB = 64
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Mantener test congelado hasta cerrar protocolo e hiperparámetros en dev
ENCODE_TEST = False
RUN_FROZEN_TEST = False

# Carpeta persistente para cachés pesadas (.npz) y checkpoints
DRIVE_STORAGE_DIR = '/content/drive/MyDrive/dm_project_artifacts' if IN_COLAB else None
STORAGE_ROOT = rq_utils.resolve_storage_root(PROJECT_ROOT, DRIVE_STORAGE_DIR)
EMBEDDING_DIR = STORAGE_ROOT / 'embeddings' / 'beto_sts_full'

RUN_ID = rq_utils.make_run_id('baseline_beto_sts')
CSV_DIR = PROJECT_ROOT / 'rq_larra' / 'csv' / 'rq_baseline' / RUN_ID
FIG_DIR = PROJECT_ROOT / 'rq_larra' / 'figuras' / 'rq_baseline' / RUN_ID
MODEL_DIR = STORAGE_ROOT / 'models' / 'baseline' / RUN_ID

for d in [CSV_DIR, CSV_DIR / 'audit', FIG_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'RUN_ID: {RUN_ID} | DEVICE: {DEVICE}')
print(f'STORAGE_ROOT (Drive/local): {STORAGE_ROOT}')
print(f'CSV_DIR: {CSV_DIR}')
print(f'FIG_DIR: {FIG_DIR}')

In [ ]:
# 3) Auditoría de integridad de particiones/
rq_utils.ensure_rama_b_on_path(PROJECT_ROOT)
from audit_splits import audit_splits
import json

summary_audit, checks_audit = audit_splits(PROJECT_ROOT / 'particiones')
summary_audit.to_csv(CSV_DIR / 'audit' / 'split_audit.csv', index=False)
(CSV_DIR / 'audit' / 'split_audit_checks.json').write_text(
    json.dumps(checks_audit, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print('Audit status:', checks_audit['status'])
assert checks_audit['status'] == 'ok', checks_audit['errors']
summary_audit

In [ ]:
# 4) Embeddings BETO-STS (reutiliza caché en Drive si ya existen)
partitions_to_encode = ('train', 'dev', 'test') if ENCODE_TEST else ('train', 'dev')
rq_utils.ensure_beto_sts_embeddings(
    PROJECT_ROOT,
    EMBEDDING_DIR,
    partitions=partitions_to_encode,
    batch_size=BATCH_SIZE_EMB,
    device=DEVICE,
)

In [ ]:
# 5) Relabeling (include same subject) + entrenamiento MLP baseline + exportación de logits/probs
results = rq_utils.run_baseline_and_export(
    PROJECT_ROOT,
    embedding_dir=EMBEDDING_DIR,
    model_output_dir=MODEL_DIR,
    csv_output_dir=CSV_DIR,
    run_id=RUN_ID,
    disorders=DISORDERS,
    seeds=SEEDS,
    neighbors=NEIGHBORS_J,
    min_control_neighbors=MIN_CONTROL_K,
    relabel_mode=RELABEL_MODE,
    max_rounds=MAX_ROUNDS,
)

# Registrar corrida y puntero latest_baseline.json para RQ1 y RQ2
rq_utils.save_latest_baseline_pointer(
    PROJECT_ROOT,
    run_id=RUN_ID,
    storage_root=STORAGE_ROOT,
    embedding_dir=EMBEDDING_DIR,
    baseline_model_dir=MODEL_DIR,
    baseline_csv_dir=CSV_DIR,
    disorders=DISORDERS,
    seeds=SEEDS,
    relabel_mode=RELABEL_MODE,
    neighbors=NEIGHBORS_J,
    min_control_neighbors=MIN_CONTROL_K,
)
rq_utils.register_run(
    PROJECT_ROOT,
    run_id=RUN_ID,
    rq='rq_baseline',
    stage='dev_baseline',
    status='completed',
    notebook='rq_larra/ipynb/00_baseline_relabeling.ipynb',
    seed=','.join(map(str, SEEDS)),
    disorder='all',
    embeddings='beto_sts_768',
    relabel_neighbors=NEIGHBORS_J,
    relabel_min_control=MIN_CONTROL_K,
    artifacts_dir=f'rq_larra/csv/rq_baseline/{RUN_ID}',
    notes=f'mode={RELABEL_MODE}; conditions=initial_labels,knn_include_same_subject',
)

results['dev_summary'][['disorder', 'condition', 'macro_f1_mean', 'macro_f1_std', 'f1_suffer_mean', 'recall_suffer_mean', 'latency_tp_fraction_mean']]

In [ ]:
# 6) Figuras diagnósticas de relabeling y baseline
rel_dis = pd.read_csv(CSV_DIR / 'relabeling_summary_by_disorder.csv')
rel_pos = pd.read_csv(CSV_DIR / 'relabeling_summary_by_position.csv')
dev_sum = results['dev_summary']

# Figura 1: Fracción de mensajes suffer relabelados por trastorno y banda temporal
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(rel_dis['trastorno'], rel_dis['share_initial_suffer_relabelled'] * 100, color=['#4C72B0', '#DD8452', '#55A868'])
axes[0].set_title('Relabeling k-NN: % mensajes suffer -> control (train)')
axes[0].set_ylabel('% mensajes relabelados')
axes[0].set_ylim(0, 100)
axes[0].grid(axis='y', alpha=0.3)

band_order = ['first_20%', '20-40%', '40-60%', '60-80%', 'last_20%']
for disorder in DISORDERS:
    sub = rel_pos[rel_pos['trastorno'] == disorder].set_index('position_band').reindex(band_order)
    axes[1].plot(band_order, sub['share_relabelled'] * 100, marker='o', label=disorder)
axes[1].set_title('Relabeling por posición temporal relativa del mensaje')
axes[1].set_ylabel('% relabelado')
axes[1].set_xlabel('Posición en historial del usuario')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'relabeling_diagnostics.png', dpi=150)
plt.show()

# Figura 2: Macro-F1 dev del baseline (Init vs k-NN include same subject)
fig, ax = plt.subplots(figsize=(8.5, 4.5))
pivot_mean = dev_sum.pivot(index='disorder', columns='condition', values='macro_f1_mean').reindex(DISORDERS)
pivot_std = dev_sum.pivot(index='disorder', columns='condition', values='macro_f1_std').reindex(DISORDERS)
pivot_mean.plot(kind='bar', yerr=pivot_std, capsize=4, ax=ax)
ax.set_title('Baseline BETO-STS + MLP en dev (Macro-F1 por sujeto, 5 semillas)')
ax.set_ylabel('Macro-F1')
ax.set_ylim(0.5, 1.02)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / 'baseline_dev_macro_f1.png', dpi=150)
plt.show()

In [ ]:
# 7) Copia de seguridad en Drive si el repo estaba en almacenamiento efímero (/content/dm_project)
if IN_COLAB and not str(PROJECT_ROOT).startswith('/content/drive'):
    backup_root = STORAGE_ROOT / 'rq_larra_backup'
    shutil.copytree(CSV_DIR, backup_root / 'csv' / 'rq_baseline' / RUN_ID, dirs_exist_ok=True)
    shutil.copytree(FIG_DIR, backup_root / 'figuras' / 'rq_baseline' / RUN_ID, dirs_exist_ok=True)
    shutil.copy2(PROJECT_ROOT / 'rq_larra' / 'csv' / 'index_runs.csv', backup_root / 'csv' / 'index_runs.csv')
    shutil.copy2(PROJECT_ROOT / 'rq_larra' / 'csv' / 'rq_baseline' / 'latest_baseline.json', backup_root / 'csv' / 'rq_baseline' / 'latest_baseline.json')
    print('Backup sincronizado en Drive:', backup_root)

In [ ]:
# 8) [SOLO CUANDO EL PROTOCOLO ESTÉ CONGELADO] Exportar predicciones de test sin reentrenar
if RUN_FROZEN_TEST:
    rq_utils.ensure_beto_sts_embeddings(
        PROJECT_ROOT, EMBEDDING_DIR, partitions=('test',), batch_size=BATCH_SIZE_EMB, device=DEVICE
    )
    test_df = rq_utils.export_frozen_test_predictions(
        embedding_dir=EMBEDDING_DIR,
        model_output_dir=MODEL_DIR,
        output_csv=CSV_DIR / 'test_message_predictions.csv',
        disorders=DISORDERS,
        seeds=SEEDS,
    )
    print('Exportadas predicciones congeladas de test:', len(test_df))